<div dir="rtl">

# HM_08 - מעבדה: בדיקות ואיכות קוד - "קוד שלא נבדק הוא השערה"

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע (4261310) ולמסלול ידע ונתונים (תשפז_ב_4261110) · מאת ד״ר ולד לנדא וד״ר רן דובין · **מפגש 8:** בדיקות ואיכות קוד

**שם:** _(כתבו כאן)_

---

## מה עושים במעבדה

במפגש 8 למדנו שקוד **"שרץ בלי שגיאה"** יכול להיות שגוי (ביקוש 2, אספקה 7, ביטחון 3: הקוד נתן 17 במקום 20), ושהכלל הוא: **קוד בלי בדיקה הוא השערה, ובדיקה שלא ראיתם נכשלת היא השערה.** במעבדה הזו תכתבו **בדיקות pytest אמיתיות** על פונקציות מהמלאי של "החנות", תראו אותן נכשלות, תתקנו את הקוד, ואז תבדקו **את הבדיקות עצמן**: האם הן תופסות גרסאות שבורות בכוונה ("מוטנטים")? ובסוף תבדקו בדיקות ש-AI כתב, ותסקרו שינוי של סוכן.

הכול רץ ב-Colab. הקבצים נכתבים לתיקייה של Colab (📁 בצד שמאל), ו-pytest רץ עליהם כמו בטרמינל.

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| א' | בדיקה ראשונה לקוד שנראה תקין | 5 בדיקות (חמש קטגוריות) ל-`calc_reorder_point`, ותיקון הקוד | 9 דק' |
| ב' | TDD: קודם בדיקות, אחר כך קוד | `test_eoq.py` ← אדום ← `calc_eoq` ← ירוק | 8 דק' |
| ג' | מי בודק את הבדיקות? מוטנטים | `is_caught`, `kill_score`, שיפור הבדיקות עד 6 מתוך 6 | 8 דק' |
| ד' | בדיקות שה-AI כתב | `find_weak_asserts`, וצעד אמיתי עם עוזר AI | 6 דק' |
| ה' | סוקרים שינוי של סוכן | `changed_files`, `removed_tests`, רשימת ממצאים והחלטה | 6 דק' |
| ו' | מסקנה | - | 2 דק' |

**מה חובה בכיתה?** חלקים **א', ב', ג'** (ליבת המפגש). אם הזמן נגמר, **ד' ו-ה'** אפשר להשלים בבית (הצעד האמיתי עם ה-AI נמצא בחלק ד').

## איך עובדים

1. **מממשים** כל מקום שמסומן `# TODO` - **בעצמכם, בלי כלי AI**. את הבדיקות כותבים בתא שמתחיל ב-`%%writefile` (הוא כותב קובץ), ואת הקוד כותבים בקובץ או בפונקציה. (רק בחלק ד' משתמשים בכלי AI, כי זה נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל שלב. כשהכול נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה לא תקין.
3. **לפני כל מדידה כותבים תחזית** בתא המיועד לכך. תחזית "שגויה" עם ניתוח טוב שווה יותר מתחזית "נכונה" בלי הסבר.
4. **עונים על שאלות הניתוח** במונחים של מה שלמדנו בהרצאה.

> **חשוב:** ציפיות בבדיקה (למשל "התוצאה צריכה להיות 20") מחשבים **ממפרט או מנוסחה, בראש או במחשבון**, ולא מהרצת הקוד. הרצת הקוד כדי לגלות מה "צריך" לצאת היא בדיוק הבדיקה המעגלית שנראה בחלק א'.

> **אל תיבהלו מהקוד.** כל פונקציה כאן היא 2-6 שורות, ובכל שלב יש הסבר ורמז. נתקעתם? קראו שוב את ה-docstring (הטקסט בין שלוש המרכאות) ובקשו עזרה מבן הזוג.

## הגשה

בסוף: **Runtime ← Restart session and run all**, ודאו שתא הבדיקה המסכם בסוף המחברת מראה `✓` לכל השלבים שביצעתם, ואז **File ← Download ← Download .ipynb** והגישו את הקובץ.

</div>

In [1]:
# תא הכנה - הריצו פעם אחת בתחילת העבודה
import os, sys, subprocess

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"       # כדי ש-Python לא ישמור גרסה ישנה של קובץ שערכתם
os.environ["PYTEST_ADDOPTS"] = "-p no:cacheprovider --color=no"

try:
    import pytest
except ImportError:                                # ב-Colab בדרך כלל pytest כבר מותקן
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    import pytest

print("✓ מוכן, pytest", pytest.__version__)

✓ מוכן, pytest 8.3.2


In [2]:
#@title תא הכנה - הקבצים של המעבדה (אפשר לא לפתוח) { display-mode: "form" }
import base64, gzip, pathlib
_FILES = {
    'reorder_v1.py': (
        'H4sIAPIO2WoC/2WQvW7EIBCEez/FyNVdcromnaUr06ZIkdbamOWEDgNawIrfPvwkThEaWO1+Mzso1ljILrOwF8UyB29cOileyakL'
        'LJOak1n5gkia0357OU8DyhnH8b0jaAhuUGTsjo7iCacKo8IwrjT3iOcflVadr0NT6sAE2ljozsjOpIjorUIo6mW0jR2rTIg5BGtK'
        '759Dm+weE/grCXVjrxGTXx54cEigCMJn1pqlAW98p2S2qhJy8Sapz42sUahZ1hwThExkfJDN/Cri5fr7De0WTlncX/hj2yPz8A0K'
        'BpPhawEAAA=='
    ),
    'inventory.py': (
        'H4sIAPIO2WoC/z2OQQvCMAyF7/sVYadNxgSPOyvoxYOC1xHabBbaxLWdoL/erdsMhBDyvfeiqQOFVrUkQ4HMI9pWk0PWFYjX5Fsl'
        'IVbwFKsN92krmwymyvP8pITFGbWgMIzI0cRPA2HwsTjADo5T32EP57LOkuqGJhA80I508l48mA6WwMlk9ZlDwARg6jGaN1VJubDr'
        'I3/mS17m08bW229p+pR2lXhxL0uOOJJOsdkPZDwif/kAAAA='
    ),
    'ai_tests.py': (
        'H4sIAPIO2WoC/5WRQWvDMAyF7/4Vgl0SKCUL6aGDDnbYYZeddg9uolCBY3my0jb79XO2ZLTQDSYwwuj5e8/2Hbxh1AgnIVX0sB/B'
        'enh6ARsjRbVeoWOBxrqmRn6HzDoH3IEesIeQRMB+2kDDItho6i3mhvrAohBGTXTTCfdA/oheWUaYhwvTGNNiIiZlLaiD+Fj7od+j'
        'ZPmDgVSCcXAKu58j2X1ZFCvYpFXmX5KUBBOTIvkpdYPZ96EVdI6t5tcm06QOHEnpiIvLjLjpAY9QXCJ0DJSE9dG6Af8fc1Hu5hda'
        '2xCEz1lZVetttV0lgdsV681V7A8UrlvsrW9/d/zLjiJ4Vnhlj5fc9DHWUZt6GHQhn0gPSzixFDFmz+cGgxL7WTLVjZsWufkEqq7C'
        'c1UCAAA='
    ),
    'mutants/m1_no_factor_2.py': (
        'H4sIAPIO2WoC/32QPQ6DMAyF95zCYoIKtZ0rOvYKXZFLDETKDzhJz9+QUikstZRIjvy95zxlFscBDIZZCCFphAH10JNba7Q2ou4l'
        'GbSyBceSuB+cDy3MTktlp9w1NwGp1AgHADq4JqbAtpfv7FaMyhM8UUd6MDuuq53bToYgQyamy7oALwJLEwb1pqr5WZaLQHf/b7AP'
        'F7pJc3FeFZpMIbLNeZz9yuGYApzK/1yOOYgPyUhteEwBAAA='
    ),
    'mutants/m2_no_sqrt.py': (
        'H4sIAPIO2WoC/32PwQqDMAyG7z5F8KQiOHYc7rhX2FUyG12hJlts9/yrzkHdYYEEEvL9f2JogB5d35E8C2QO6DpDE7KpQdSQdr3M'
        'voa7OGN5XLvylEEMO8AOgBYOkUmwZfLZXULRzgRXdIEuqqJFvnFLrhCs0BRiYfFwI2Aa0dsX5eXXMj0E2vN/g2050Y2aD5ltoqnk'
        'gzIcofp5p0o/aXbG2RuPe9loNwEAAA=='
    ),
    'mutants/m3_no_holding_check.py': (
        'H4sIAPIO2WoC/12OwQ6CMAyG73uKhhMQgsaj8eoreCWVFViydVI6n1+YHsA/aZM2/f7+LryiKATUyRhjaYAefd9RnEtkTug7SwHZ'
        'NhDFknR9XLSBKXrreMxTdTWwyg1wAOAG55XZYdvme7tJ0C0ED/SJ7iJRyuLHbZUhyFBIa+Oo8CRgGlHdm4oq2whpEs7Z22UWLS9Q'
        '/4Wo9/9Px9zmAxluEYz8AAAA'
    ),
    'mutants/m4_zero_demand_error.py': (
        'H4sIAPIO2WoC/32QzwqDMAzG732K4ElFNtlxuONeYVfpbNRC/2ja7vlXOwd1hwUSCOT3fUmkXix50NzPjDGBIwxcDT3ateTGBK56'
        'gZob0YAlgdQP1vkGZquENFPqqiuDGHKEAwDdDdoIZRx00H6GtyAuHcKDq4B3IktlsYNbJggSpEMsxnp4IhicuJcvLKqvZ75Jsvxn'
        'sA9nulFzsU5mmoQ+kEkPObmVfHmB+ueyOr/pfHwGewNfVX+QUQEAAA=='
    ),
    'mutants/m5_hardcoded_example.py': (
        'H4sIAPIO2WoC/42Qwa6CMBBF9/2KG1fwQhSRF9DI0l9wS/roKE2g1aH4/RbUpGxenGSadDrn3s7o/mbZoZeuFUIouqCRXVOTvUfS'
        'mFF2taJeGpXAsiKuGzu4BK3tlDbX+RYfBHzoCxYAjkg9E2BT5dU7BUs9EM6yG+nEbDlavbkpZwgz1I/+MNbhj2DoKp1+0Cr+WIYf'
        'wbH63+DdHOh6zZsd9FLz68FRVYi2WZom+PWZxYE5uZENsjxf7/NyX+RZUe62RSmCx2nn6+HObmmIn3Bnm6WleALE6GgvsAEAAA=='
    ),
    'mutants/m6_no_negative_check.py': (
        'H4sIAPIO2WoC/1WOwQrCMBBE7/mKpadUiopH0aO/4DWszdYGkmy7Sfx+axQkcxiY3Rl4LiwsGQLmWSllaYIR/WiIV40xFvTGUsBo'
        'B2CxJGbklAeY2VsXnzX1ZwWb3NRc4XKF4/fzkaBLBHf0hW4iLLr7laGWQ9nsQbBwctm9qOvrUigXiRVun1bJ+gQ7aLC2/OeCQwum'
        '3ktlkIDdAAAA'
    ),
    'kit/reorder_buggy.py': (
        'H4sIAPIO2WoC/2WQvW7EIBCEez/FyNVdcromnaUr06ZIkdbamOWEDgNawIrfPvwkThEaWO1+Mzso1ljILrOwF8UyB29cOileyakL'
        'LJOak1n5gkia0357OU8DyhnH8b0jaAhuUGTsjo7iCacKo8IwrjT3iOcflVadr0NT6sAE2ljozsjOpIjorUIo6mW0jR2rTIg5BGtK'
        '759Dm+weE/grCXVjrxGTXx54cEigCMJn1pqlAW98p2S2qhJy8Sapz42sUahZ1hwThExkfJDN/Cri5fr7De0WTlncX/hj2yPz8A0K'
        'BpPhawEAAA=='
    ),
    'kit/reorder_ref.py': (
        'H4sIAPIO2WoC/0WNMQ6CQBREe04xoQKlMLEzWnoFW/JlB7MJ7JK/f0m8vSISppvJvDzHHp0MXauM6qjtFH2wynGU4BoMFNeaH9kg'
        'SU973871pcA3vsd6whUnRN2v27ACS1uJJSo+EQ8ZMu+qUavShylbwpiTIUTDkwh8ifmZZf3jlJY1bLYDqt10/Evq4gNjV2ZryAAA'
        'AA=='
    ),
    'kit/reorder_no_negcheck.py': (
        'H4sIAPIO2WoC/0tJTVNITsxJji9KzS9KSS2KL8jPzCvRSEnNTcxL0VHISU1MiS/JzE3VUShOTEstqbQ11rTiUgCCotSS0qI8BYhC'
        'BS0FDbhSBW2oWk0uADGkpTVeAAAA'
    ),
    'kit/reorder_default5.py': (
        'H4sIAPIO2WoC/0WNMQ6CQBREe04xoQKlsLExWnoFW/JlB7MJ7JK/f0m8vSISppvJvDzHHp0MXauM6qjtFH2wynGU4BoMFNeaH9kg'
        'SU973871pcA3vsd6whUnRN2v27ACS1uJJSo+EQ8ZMu+qUavShylbwpiTIUTDkwh8ifmZZf3jlJY1bLYDqt10/Evq4gPDmzSJyAAA'
        'AA=='
    ),
    'kit/inventory_stub.py': (
        'H4sIAPIO2WoC/z2OQQvCMAyF7/sVYadNxgSPOyvoxYOC1xHabBbaxLWdoL/erdsMhBDyvfeiqQOFVrUkQ4HMI9pWk0PWFYjX5Fsl'
        'IVbwFKsN92krmwymyvP8pITFGbWgMIzI0cRPA2HwsTjADo5T32EP57LOkuqGJhA80I508l48mA6WwMlk9ZlDwARg6jGaN1VJubDr'
        'I3/mS17m08bW229p+pR2lXhxL0uOOJJOsdkPZDwif/kAAAA='
    ),
    'kit/inventory_ref.py': (
        'H4sIAPIO2WoC/32QQWvDMAyF7/4VwqeklHbsWLrbCrsPdg2arbaGWGoUe9B/XyfphnOZQAaZp/fxFOJNNEHEdDXGeDqDw951JEOD'
        'zBn7zlNE9lsQ9aSdkzFt4Sq9D3yZp/ZgoJS19uSEJQa3SGHIyCmk+wHGQVPzCht4L/0Je/hod0U/74UzrEBwhJdiUOGmn4UxlWIY'
        'Cb6wz3RSFW3sc2/qBTwvxVwelgTfBEwXTOGHbPuLrAPA8e1/wFNc+RbPm4yh8lRKWXm+4+4v7jrYps60X9/QPABA3Wv7iAEAAA=='
    ),
    'kit/oracle_reorder.py': (
        'H4sIAPIO2WoC/62RTUvEMBCG7/kVc+xCK2lFBWGPe/UkXkN2M+0G8lEmaaX+elPtgtGqBZ1TSCbPM3mjbe8pQj9FDJG15C0QelJI'
        'YqxBv5+epDmJy3bvtYuMMYUtzJeE82SlKXb3DFLJEHD1StGUcFfC9Q72e2j4V4Lw8Zya3WCPSOF33k0JNS+heQPeZsCj7oRCK53a'
        'MBbn2WA8I70g+c2oj6CMkhZyMFEE2WKctkW1lhN2MuoRPw30rON5+cErkjpgKJ6kGfBA5GnpmWtFVNXLyKsag1KJqC3+gyk9qfre'
        'lAfzN02yVLnG6hC068Q4Q36QPE79FseDd3iJ7RVXM6YXQAMAAA=='
    ),
    'kit/oracle_eoq.py': (
        'H4sIAPIO2WoC/62QwW6DMAyG7zxFjiAVFFKC1Eoc9wq7ogxciAQxddKu7OkHHUxFhcO0Rsoldr7P/nXbITnW9Q6s806ELdPmCsYh'
        '9Uz/FAvVFDng2fO8Ek5s7Mxd3+nh3Q+OHhuOshYeOv1YcL5jcrgiYFk28SPVdYQ3XyRJdEgOO0bQZDGE++AR3SiqYBPM+UiO7/gV'
        'dLoXUSLTDTTcVOFye74o2jTIu0DIQbLCl2NxYqcL9hcQ5iW0ypRb6EUk/OlzjU2pTZUXaN2M+NSunkcgpS1Y/101F3gjQpp6xrOS'
        'PF+MZ6BSTl/h1ZZQrGuWSfxZEMo5q1U6Ugn0ihXCZ8kHuvrX9K8FZvg3o9KtUGUDAAA='
    ),
    'lab_c/agent_change.diff': (
        'H4sIAPIO2WoC/51UTW/bMAw9z7+C8EmeP2IbabsGNZBi67DDDju0uwSBoVpyIsyRVElul/360R9pkzRoh8WxYVPkI6n3RCbqGmLT'
        'wj2vleETIR+5dMpsE70FWjtuDkxeHMenXD/kaX4eZ2mcnUOezvJPs/TCC8PwBMRJ1/kc4mwaTSHsn/O5B7ufocJy+Emblt8Yowzx'
        'Gd9QyaC7lWHcQKWsg02LD6kclgeSr6gTj9wPBhxRw1o1TMhV2fteFZDO3koxeu8hI6pWVoyocR/GXWskbKhbJ/bBOJLDR6xKtrQp'
        'xxo/DhUOWScHRQReuAdiVCsZ+V+owGNHPBqulXEvJD5/7zP4bHyHvrf9eu6iLEXuojzvuAvFpgsAZT0YXzW2QC3gXzMPvPDLzdfr'
        'u++35Y/r229QgP95Nrmz3NiJoXLSN2qTyj766BszXkOjKCsHO9G4S8HMC73w1EqxDx2MLI+7rFliOLoj8oCC8Hh1ME7pUhvF2spZ'
        'wuoIZHG2i2Y1lsjqhForVpIYjmpueYGWh5ZKJ9wW2cGvVgqHIKLiwUFaXFohwfp+S/wxRynpBoWUjFiJbTckSGRDzYpbR2TQtbdr'
        '8Km0TlW/SP+MUCu6a39IYNvGYXWL5WBAXkGAQEFRueKk4XKICnYR43HojYloVLUQy4WvZLlGhvwlXHXwi6PlXdECPZbYa5Zke3Av'
        'hSRUa446fiN8lH0rW8sZFt5VOAQfHYje9krXDnenPDGkXtn3df5q8R29/5t/r/tR+Fl0fjC0KtpUpeHDidVKSEfwjMDFILhe0X0S'
        'rh5Kt9UC/Qky1I8VVBnHE9ND4DrJ8jSN4AzvPICiAL3tQrutNuo3yafT5HJ6GQG9t0WapBkOJ7zgIMUfblS5PzHITtpPwq13iP0Y'
        'tORlDgazo44Oy0kD7y+5TEw7PwYAAA=='
    ),
    'lab_c/before/inventory.py': (
        'H4sIAPIO2WoC/4WSQWuDMBiG7/6KF0/alnZst9LeWth5g10lM59tQBNNPgf++5no2rhBF8ghku958r6omtZYRiP4miSJpAqlqMvC'
        'krGSbNEapTmTQtVDIakRWm5Qk5AFq4YKKQa3gRMV8RAOx5d8n2BcaZq+TQgEBI4IEEwQrJB5DDwG6xkBj8i342xgqAqxGAc8wdhf'
        'ev91UvplhXKED1H3dLbW2CxVuu3ZoekdQxvGJ0HTRbD6ojQPc5a4t3qpmp8XadZxzDyuikyXCa17Ud8amrorjeMNrqaWSl/C6V7O'
        'uTTaNKqcrqLrhWbFwx6us5w9jw84jfsdO7wuClmIfhq56/5rY57zexKHoQfljMo4AA7Hx4L5csQdma1x6m/h/pfb3uIug63iTLtl'
        'h8k3AjQ5D7MCAAA='
    ),
    'lab_c/before/report.py': (
        'H4sIAPIO2WoC/02NOwrDMAyGd5/CdLJL8dalkLMENVJSQyK7shzI7avSDBVCg77/kbdaRH0FRmjetqJzDmn2awEciyBJCxX0FR/O'
        '2whpFzZZEjLB1PYfPV1a6lilYJ+0BZxvnof76cTZD3YStJYXDkI7cafBPu8OrFkPf/3yzlktI08U/xuNLFJ6fR7hcjaMDBtdYjqj'
        'UutbiIlXkIWaBo7uAxxjuwrfAAAA'
    ),
    'lab_c/before/test_inventory.py': (
        'H4sIAPIO2WoC/51PTWvDMAy9+1fomIAXHC9jtJDj/sKuxkvUVpBanux1y3793DWFhXWXCQSC9/Q+6BhZMsQ5Y8pqJ3wECicMmWUG'
        'uoCDnwaH/KovlyDLiOIiU8hKqRF3cP5eA1W9VVDGp4RXjTXBanjUcF9D34M1fwq5gHuf6YRXxXfKhyVwI54SpurZT2/4JMKycM5z'
        'w/KuLZ71T6tSy+U5UiHfSlzgqrXGaHgoa7+zLtY+RuGPynZds+k2GvxL6k1j2l/ynyjsDjyNFPZu4JT/VWSdxNTqC++T6xa6AQAA'
    ),
    'lab_c/after/inventory.py': (
        'H4sIAPIO2WoC/4WSsWrDMBRFd3/FxZOdhKS0W0i2BDq30NWo1ksisCVHeir472vJbiK3kAo0yOido3uxajtjGa3gS5Zlkk6oRVNX'
        'loyVZKvOKM2FFKrpK0mt0HKFhoSsWLVUSdG7FZw4EffxsH8ptxmGlef524hARGCPCMEIwQJFwCBgsJwQCIhyPcxGhjohFWOHJxj7'
        'Sx++jsqwrFCO8CEaT0drjS1ypTvPDq13DG0YnwRNZ8Hqi/Iyzllib/VcNT0v0SzTmGVaFZlrIbT2ork1NHZXG8crXEwjlT7H072c'
        'Y220aVU9XsXVC82K+y3c1XLxPDzgMOx3bPA6K2Qm+mnkrvuvjWku7FEchx6UMyjTANjtHwumywl3YHbGqb+FW+O1LMKPt76Fnsdb'
        'pMk28ybL7Bt/9TalugIAAA=='
    ),
    'lab_c/after/report.py': (
        'H4sIAPIO2WoC/31QPU/DMBDd/SusTA6qXHVgqchQFRADA0M7RVFk4ktqkZ6NP4r677l80BYGLMs637337t2Zo7M+chuYmSKnUKvA'
        '6TrN2OPT82b/uqvfNrsXXvBsu17uA/iw9AqX1msKZRNOGWNMQ8t7q3Q9pYVT8VDc8vM143Q8xOSR1KUHQhN7hOazRLSudt7q1MQg'
        'dLvgWNzPTN2SBd1KFYLpUHg4ASYoKPOZFEYTz/xuqCc0kTRMA/ltR6p03ib3fhbZ3KFGdYQsl7OUDOkocom98h2EKDC/zPVVh2ib'
        'DzG+C+6tu4wTUh/JWFmN/9Z6brhBThvqQPSAEyef8cMxLR9z0vS2KU1VZhbrAy0+q/jDoF3+Kf/YNYSoaMiVXF3Vri6kcg5Qi3/Y'
        '00YSpgCaTA/2Ju6vVU0p9g02vAgJHQIAAA=='
    ),
    'lab_c/after/test_inventory.py': (
        'H4sIAPIO2WoC/52PwQqDMAyG732KHC24UR1jMNhxr7BrKRq1oI1LO4d7+lVUmLBd9kMg8Cf/l9iuJw7QjwF9EBVTB9YN6ALxCHY2'
        'C9MWGumezh0jcYmse7IuCCFKrGDa3hqJPAuIMt7jmrEdyFM4pXCQcLlArn4GaYe1CXbANfFpQ7McvGdjPfrkZtoHXpmJl5lJX5C7'
        'LDLlJyq+pV/IpBtqS+tqXZAPf5FiUpLlSqVwjKWkeAPxn+GjWwEAAA=='
    ),
}
for _name, _b64 in _FILES.items():
    pathlib.Path(_name).parent.mkdir(parents=True, exist_ok=True)
    pathlib.Path(_name).write_bytes(gzip.decompress(base64.b64decode(_b64)))
print('✓ נוצרו', len(_FILES), 'קבצים: reorder_v1.py, inventory.py, ai_tests.py, mutants/, lab_c/, kit/')

✓ נוצרו 24 קבצים: reorder_v1.py, inventory.py, ai_tests.py, mutants/, lab_c/, kit/


<div dir="rtl">

> **על התיקייה `kit/`:** בצד שמאל (📁) נוצרה תיקייה בשם `kit` עם הקוד הנכון ובדיקות הייחוס של המורה - תאי הבדיקה שבמחברת משתמשים בהם כדי לשפוט את העבודה שלכם. **אל תפתחו אותה**: כך תאי הבדיקה נשארים הוגנים, ואתם לומדים יותר.

</div>

In [3]:
#@title תא עזר - מריץ pytest ומדווח (אפשר לא לפתוח) { display-mode: "form" }
import glob, os, re, shutil, subprocess, sys, tempfile


def _run_pytest_in(cwd, target):
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", "--color=no", target],
                       cwd=cwd, capture_output=True, text=True, timeout=120)
    out = r.stdout + r.stderr
    lines = [l for l in out.strip().splitlines() if l.strip()]
    last = lines[-1] if lines else ""
    def count(word):
        m = re.search(r"(\d+) " + word, last)
        return int(m.group(1)) if m else 0
    return {"passed": count("passed"), "failed": count("failed"), "errors": count("error"), "out": out, "summary": last}


def run_tests(test_file, impl_file, module="inventory"):
    """Runs pytest on test_file against impl_file, which is copied (as <module>.py) to a clean temporary folder.
    Returns a dict: passed, failed, errors (numbers), out (the full text), summary (the last line)."""
    with tempfile.TemporaryDirectory() as d:
        shutil.copy(impl_file, os.path.join(d, module + ".py"))
        shutil.copy(test_file, os.path.join(d, "test_copy.py"))
        return _run_pytest_in(d, "test_copy.py")


def run_dir(folder):
    """Runs pytest inside a folder (for lab C)."""
    return _run_pytest_in(folder, ".")


def failed_names(result):
    return [l.split("::")[1].split(" ")[0] for l in result["out"].splitlines() if l.startswith("FAILED") and "::" in l]


print("✓ הפונקציות העזר מוכנות")

✓ הפונקציות העזר מוכנות


<div dir="rtl">

---

# חלק א' - בדיקה ראשונה לקוד שנראה תקין (9 דק')

## הרקע

בפתיחת ההרצאה ראינו פונקציה שנראית בסדר ורצה בלי שגיאה, אבל נתנה 17 במקום 20. כאן היא, מהחנות: **נקודת הזמנה** - כשהמלאי יורד עד אליה, מזמינים מהספק.

> נקודת הזמנה = ביקוש יומי × (זמן אספקה בימים + ימי ביטחון)

הקובץ `reorder_v1.py` כבר נמצא בתיקייה. קראו אותו:

</div>

In [4]:
print(open("reorder_v1.py").read())

def calc_reorder_point(demand, lead_time, safety=3):
    """Reorder point = daily demand * (lead time in days + safety days).

    demand: average units sold per day
    lead_time: supplier lead time in days
    safety: extra days of stock kept as a buffer
    Negative inputs are invalid and must raise ValueError.
    """
    return demand * lead_time + safety



<div dir="rtl">

**לפני שמריצים:** האם השורה האחרונה תואמת את ה-docstring? מה תחזיר הפונקציה עבור ביקוש 2, אספקה 7 וביטחון 3, ומה התוצאה לפי הנוסחה?

**התחזית שלי:** לפי הנוסחה: 2 × (7 + 3) = 20. הקוד מחשב demand * lead_time + safety, כלומר 2 × 7 + 3 = 17: ימי הביטחון נוספו כמספר בודד ולא הוכפלו בביקוש. קשה לראות את זה בקריאה ראשונה כי הסוגריים חסרים.

</div>

In [5]:
import importlib, reorder_v1
calc_reorder_point = importlib.reload(reorder_v1).calc_reorder_point    # reload: טוען מחדש את הקובץ אחרי שינוי

print("הקוד החזיר:", calc_reorder_point(2, 7, 3))
print("לפי הנוסחה:", 2 * (7 + 3))

הקוד החזיר: 17
לפי הנוסחה: 20


<div dir="rtl">

## שלב 1 - בדיקה שנכתבה "מהקוד"

כך נראית **בדיקה מעגלית**: מישהו (או ה-AI) הריץ את הקוד, ראה `17`, וכתב `assert ... == 17`. הציפייה לקוחה מהקוד ולא מהמפרט. הריצו את שני התאים (הראשון כותב קובץ בדיקה, השני מריץ pytest, בדיוק כמו בטרמינל):

</div>

In [6]:
%%writefile test_circular.py
# a test written from the code: the expected value 17 was copied from running the code
from reorder_v1 import calc_reorder_point


def test_basic():
    assert calc_reorder_point(2, 7, 3) == 17

Writing test_circular.py


In [7]:
!python -m pytest -q test_circular.py

.                                                                        [100%]


1 passed in 0.18s


<div dir="rtl">

**(א)** pytest הראה ירוק (`1 passed`), והקוד שגוי. מה חסר בבדיקה הזו? מי צריך להגדיר את הערך `20`, ומאיפה הוא מגיע?

**תשובה:** הבדיקה אישרה מה שהקוד עושה ולא מה שהוא צריך לעשות: הציפייה 17 נלקחה מהקוד עצמו, ולכן הלולאה לא כוללת אדם ולא מפרט. את הציפייה הנכונה (20) מגדיר אדם, ממפרט או מנוסחה (2 × 10). ירוק אומר רק שהקוד עושה מה שהבדיקה מצפה, לא שהוא נכון.

</div>

<div dir="rtl">

## שלב 2 - חמש בדיקות, אחת מכל קטגוריה

כתבו קובץ בדיקות אמיתי, **מהמפרט** (הנוסחה וה-docstring), לא מהרצת הקוד. לכל פונקציה שואלים חמש שאלות:

| # | קטגוריה | המקרה | מה מצפים |
|---|---|---|---|
| 1 | רגיל | ביקוש 2, אספקה 7, ביטחון 3 | `20` |
| 2 | גבול | ביקוש 0 | `0` |
| 3 | ריק או חסר | `None` במקום ביקוש | `TypeError` |
| 4 | לא חוקי | ביקוש `-1` | `ValueError` (כתוב ב-docstring) |
| 5 | כלל עסקי | בלי ימי ביטחון: `calc_reorder_point(2, 7)` | `20` (ברירת המחדל היא 3) |

**כלים:**
- בדיקת ערך: `assert calc_reorder_point(2, 7, 3) == 20`
- בדיקה שחייבת לזרוק שגיאה:
```python
with pytest.raises(ValueError):
    calc_reorder_point(...)
```
(הקריאה לפונקציה נכתבת **בתוך** ה-`with`; הבדיקה עוברת רק אם הייתה שגיאה מהסוג הזה.)

בכל בדיקה: שורה אחת (או שתיים). **אל תריצו את הקוד כדי לגלות מה הציפייה.**

</div>

In [8]:
%%writefile test_reorder.py
import pytest
from reorder_v1 import calc_reorder_point


def test_normal():
    # 1 normal: demand 2, lead time 7, safety 3, expect 20
    assert calc_reorder_point(2, 7, 3) == 20


def test_zero_demand():
    # 2 boundary: demand 0, expect 0
    assert calc_reorder_point(0, 7, 3) == 0


def test_missing_value():
    # 3 missing: demand None, expect TypeError  (use pytest.raises)
    with pytest.raises(TypeError):
        calc_reorder_point(None, 7, 3)


def test_negative_demand():
    # 4 invalid: demand -1, expect ValueError  (use pytest.raises)
    with pytest.raises(ValueError):
        calc_reorder_point(-1, 7, 3)


def test_default_safety():
    # 5 business rule: calc_reorder_point(2, 7) without the 3rd argument, expect 20
    assert calc_reorder_point(2, 7) == 20

Writing test_reorder.py


In [9]:
# בדיקה - חלק א', שלב 2: הבדיקות שכתבתם
def check_a1():
    src = open("test_reorder.py", encoding="utf-8").read()
    assert src.count("def test_") >= 5, "test_reorder.py: צריכות להיות חמש בדיקות (פונקציות שמתחילות ב-test_), אחת מכל קטגוריה"
    good = run_tests("test_reorder.py", "kit/reorder_ref.py", "reorder_v1")
    if "NotImplementedError" in good["out"]:
        raise NotImplementedError
    assert good["failed"] == 0 and good["errors"] == 0 and good["passed"] >= 5, \
        "הבדיקות שלכם נכשלות גם על הקוד הנכון של המורה (נכשלו: " + ", ".join(failed_names(good)) + \
        "). כנראה הציפייה שגויה: בדקו מול הנוסחה ביקוש × (אספקה + ביטחון)"
    bad = run_tests("test_reorder.py", "kit/reorder_buggy.py", "reorder_v1")
    assert bad["failed"] >= 3, f"הבדיקות שלכם צריכות להיכשל על הקוד השגוי (לפחות 3 כשלים), ונכשלו {bad['failed']} בלבד. בדקו שהציפיות נגזרות מהנוסחה"
    no_neg = run_tests("test_reorder.py", "kit/reorder_no_negcheck.py", "reorder_v1")
    assert no_neg["failed"] >= 1, "הבדיקות שלכם לא תפסו גרסה עם נוסחה נכונה אבל בלי בדיקת קלט שלילי: חסרה בדיקה מקטגוריית 'לא חוקי' (pytest.raises(ValueError))"
    def5 = run_tests("test_reorder.py", "kit/reorder_default5.py", "reorder_v1")
    assert def5["failed"] >= 1, "הבדיקות שלכם לא תפסו גרסה שבה ברירת המחדל של ימי הביטחון שגויה: חסרה בדיקה מקטגוריית 'כלל עסקי' (קריאה בלי הפרמטר השלישי)"
    assert "TypeError" in src, "test_reorder.py: חסרה בדיקת 'ריק או חסר' (None ← TypeError)"
    return True

check_a1() and print("✓ חלק א' שלב 2: חמש בדיקות, עוברות על הקוד הנכון ונכשלות על השגוי")

✓ חלק א' שלב 2: חמש בדיקות, עוברות על הקוד הנכון ונכשלות על השגוי


<div dir="rtl">

**לפני שמריצים את הבדיקות שלכם על `reorder_v1.py`:** כמה מחמש הבדיקות ייכשלו, ואיזו מהן תעבור? (חשבו על כל אחת: מה הקוד השגוי עושה בה?)

**התחזית שלי:** ייכשלו ארבע ותעבור אחת. test_normal: 17 במקום 20. test_zero_demand: 0×7+3 = 3 במקום 0. test_default_safety: 17 במקום 20. test_negative_demand: הקוד לא בודק קלט שלילי ולא זורק שגיאה. test_missing_value עוברת גם על הקוד השגוי, כי None * 7 זורק TypeError בפייתון בכל מקרה.

</div>

In [10]:
!python -m pytest -q test_reorder.py

FF.FF                                                                    [100%]
=================================== FAILURES ===================================
_________________________________ test_normal __________________________________

    def test_normal():
        # 1 normal: demand 2, lead time 7, safety 3, expect 20
>       assert calc_reorder_point(2, 7, 3) == 20
E       assert 17 == 20
E        +  where 17 = calc_reorder_point(2, 7, 3)

test_reorder.py:7: AssertionError
_______________________________ test_zero_demand _______________________________

    def test_zero_demand():
        # 2 boundary: demand 0, expect 0
>       assert calc_reorder_point(0, 7, 3) == 0
E       assert 3 == 0
E        +  where 3 = calc_reorder_point(0, 7, 3)

test_reorder.py:12: AssertionError
_____________________________ test_negative_demand _____________________________

    def test_negative_demand():
        # 4 invalid: demand -1, expect ValueError  (use pytest.raises)
>       with pytest.

=========================== short test summary info ============================
FAILED test_reorder.py::test_normal - assert 17 == 20
FAILED test_reorder.py::test_zero_demand - assert 3 == 0
FAILED test_reorder.py::test_negative_demand - Failed: DID NOT RAISE <class '...
FAILED test_reorder.py::test_default_safety - assert 17 == 20
4 failed, 1 passed in 0.14s


<div dir="rtl">

**(ב)** קראו את הפלט: ליד כל כשל כתוב `assert X == Y` - מה הוא `X` ומה `Y`? אחת הבדיקות עברה על קוד שגוי - האם היא חסרת ערך? ומה מלמדים על הקוד שני הכשלים הראשונים לעומת השלישי (`DID NOT RAISE`)?

**תשובה:** משמאל ל-== מה שהפונקציה החזירה ומימין מה שציפינו. הכשלים של test_normal ושל test_zero_demand באים מהנוסחה (ימי הביטחון לא הוכפלו בביקוש), ואילו DID NOT RAISE אומר שחסרה בדיקת קלט שלילי. test_missing_value לא תפסה את הבאג, אבל היא מתעדת התנהגות רצויה ושומרת שלא נשבור אותה בעתיד: לא כל בדיקה צריכה לתפוס את הבאג הנוכחי. כאן הבדיקות נכונות והקוד שגוי, וזה בדיוק המצב הרצוי.

</div>

<div dir="rtl">

## שלב 3 - מתקנים את הקוד

עכשיו, כשהבדיקות אדומות, מתקנים את **הקוד** (לא את הבדיקות!). הקובץ `reorder_v1.py` נכתב מחדש בתא הבא. ממשו שני דברים בגוף הפונקציה:

1. אם אחד משלושת הקלטים שלילי - `raise ValueError("...")` (הודעה באנגלית, כרצונכם).
2. החזירו `demand * (lead_time + safety)` - **עם הסוגריים**.

**כלי:** תנאי עם `or`: `if demand < 0 or lead_time < 0 or safety < 0:`

</div>

In [11]:
%%writefile reorder_v1.py
def calc_reorder_point(demand, lead_time, safety=3):
    """Reorder point = daily demand * (lead time in days + safety days).

    demand: average units sold per day
    lead_time: supplier lead time in days
    safety: extra days of stock kept as a buffer
    Negative inputs are invalid and must raise ValueError.
    """
    if demand < 0 or lead_time < 0 or safety < 0:
        raise ValueError("inputs must not be negative")
    return demand * (lead_time + safety)

Overwriting reorder_v1.py


In [12]:
!python -m pytest -q test_reorder.py

.....                                                                    [100%]


5 passed in 0.13s


In [13]:
# בדיקה - חלק א', שלב 3: התיקון
def check_a2():
    mine = run_tests("test_reorder.py", "reorder_v1.py", "reorder_v1")
    if "NotImplementedError" in mine["out"]:
        raise NotImplementedError
    assert mine["failed"] == 0 and mine["errors"] == 0, "הבדיקות שלכם עדיין נכשלות על הקוד המתוקן: " + ", ".join(failed_names(mine))
    oracle = run_tests("kit/oracle_reorder.py", "reorder_v1.py", "reorder_v1")
    assert oracle["failed"] == 0 and oracle["errors"] == 0, \
        "בדיקות הייחוס של המורה נכשלו על הקוד שלכם: " + ", ".join(failed_names(oracle)) + " (רמז: האם בדקתם את שלושת הקלטים? האם הסוגריים במקום?)"
    return True

check_a2() and print("✓ חלק א' שלב 3: הקוד מתוקן וכל הבדיקות ירוקות")

✓ חלק א' שלב 3: הקוד מתוקן וכל הבדיקות ירוקות


<div dir="rtl">

---

# חלק ב' - TDD: קודם בדיקות, אחר כך קוד (8 דק')

## הרקע

בחלק א' הקוד כבר היה קיים. ב-**TDD** (פיתוח מונחה בדיקות) הסדר הפוך, במעגל של שלושה שלבים:

1. **אדום** - כותבים בדיקה והיא נכשלת (עוד אין קוד).
2. **ירוק** - כותבים את המימוש הפשוט ביותר שגורם לה לעבור.
3. **שיפור** - מסדרים בלי לשבור (נעשה זאת במפגש 11).

למה לראות אדום? **בדיקה שמעולם לא נכשלה אינה מוכיחה שהיא בודקת משהו** (בדיקה יכולה לעבור גם כשהיא ריקה). אדום ואז ירוק מוכיח שהמימוש הוא שגרם לשינוי.

הפונקציה: **כמות הזמנה כלכלית (EOQ)** - כמה יחידות כדאי להזמין בכל הזמנה כדי למזער עלויות.

> EOQ = שורש של (2 × D × S ÷ H), כאשר D = ביקוש שנתי, S = עלות הזמנה אחת, H = עלות החזקה ליחידה בשנה.

הכללים (המפרט): ביקוש או עלות הזמנה **שליליים** ← `ValueError`. עלות החזקה **אפס או שלילית** ← `ValueError`. ביקוש **אפס** ← התוצאה 0.

</div>

<div dir="rtl">

## שלב 1 - טבלת מקרים (מהמפרט, לא מהקוד)

האדם מגדיר את המקרים והתוצאות, וה-AI (או אתם) מתרגם אותם לקוד. הטבלה שלנו (הערכים מחושבים מהנוסחה: למשל √(2×1200×50÷2) = √60000 ≈ 244.95):

| קטגוריה | ביקוש D | עלות הזמנה S | עלות החזקה H | תוצאה צפויה |
|---|---|---|---|---|
| רגיל | 1200 | 50 | 2 | בערך 244.95 |
| רגיל | 10000 | 100 | 5 | בערך 632.46 |
| גבול | 0 | 50 | 2 | 0 |
| לא חוקי | 1200 | 50 | **0** | `ValueError` |
| לא חוקי | **-5** | 50 | 2 | `ValueError` |

(קטגוריית "ריק או חסר": המפרט לא קובע התנהגות, ולכן אין לנו מקרה כזה.)

**כלים:** מספרים עשרוניים לא משווים עם `==` מדויק, אלא עם סטייה מותרת: `assert result == pytest.approx(244.949, rel=1e-3)` (סטייה של עד אלפית). הבדיקה הראשונה כבר כתובה כדוגמה. ל-`ValueError` משתמשים באותו `with pytest.raises(ValueError):` מחלק א'.

## שלב 2 - כותבים את הבדיקות (עדיין אין קוד!)

</div>

In [14]:
%%writefile test_eoq.py
import pytest
from inventory import calc_eoq


def test_typical():
    # 1 normal (example): D=1200, S=50, H=2, expect about 244.949
    assert calc_eoq(1200, 50, 2) == pytest.approx(244.949, rel=1e-3)


def test_large():
    # 1 normal: D=10000, S=100, H=5, expect about 632.456  (use pytest.approx)
    assert calc_eoq(10000, 100, 5) == pytest.approx(632.456, rel=1e-3)


def test_zero_demand():
    # 2 boundary: D=0, expect 0
    assert calc_eoq(0, 50, 2) == 0


def test_zero_holding_cost():
    # 4 invalid: H=0, expect ValueError
    with pytest.raises(ValueError):
        calc_eoq(1200, 50, 0)


def test_negative_demand():
    # 4 invalid: D=-5, expect ValueError
    with pytest.raises(ValueError):
        calc_eoq(-5, 50, 2)

Writing test_eoq.py


In [15]:
# בדיקה - חלק ב', שלב 2: הבדיקות כתובות, וסביבת "אדום" תקינה
def check_b1():
    src = open("test_eoq.py", encoding="utf-8").read()
    assert src.count("def test_") >= 5, "test_eoq.py: צריכות להיות לפחות חמש בדיקות"
    ref = run_tests("test_eoq.py", "kit/inventory_ref.py")
    if "NotImplementedError" in ref["out"]:
        raise NotImplementedError
    assert ref["failed"] == 0 and ref["errors"] == 0 and ref["passed"] >= 5, \
        "הבדיקות נכשלות גם על הקוד הנכון של המורה (נכשלו: " + ", ".join(failed_names(ref)) + "). בדקו את הציפיות מול הטבלה"
    red = run_tests("test_eoq.py", "kit/inventory_stub.py")
    assert red["passed"] == 0 and red["failed"] + red["errors"] == ref["passed"], \
        "על קוד ריק (stub) כל הבדיקות צריכות להיכשל (אדום). בדקו שאין בדיקה שעוברת בלי לקרוא לפונקציה"
    return True

check_b1() and print("✓ חלק ב' שלב 2: הבדיקות כתובות ונכונות")

✓ חלק ב' שלב 2: הבדיקות כתובות ונכונות


<div dir="rtl">

## שלב 3 - אדום

הקובץ `inventory.py` עדיין **ריק**: הפונקציה `calc_eoq` רק זורקת `NotImplementedError`. הריצו את הבדיקות: צריך לראות **הכול אדום**.

</div>

<div dir="rtl">

**לפני ההרצה:** כמה מהבדיקות ייכשלו על הקוד הריק? גם בדיקות ה-`ValueError` (שמצפות לשגיאה)? הסבירו.

**התחזית שלי:** כולן ייכשלו. בדיקות הערך נכשלות כי הפונקציה זורקת שגיאה במקום להחזיר מספר. גם בדיקות ה-ValueError נכשלות, כי pytest.raises(ValueError) מקבל רק ValueError, והפונקציה זורקת NotImplementedError. האדום הזה הוא הצלחה: הוא מוכיח שהבדיקות רצות ושלא עוברות על קוד ריק.

</div>

In [16]:
!python -m pytest -q test_eoq.py

FFFFF                                                                    [100%]
=================================== FAILURES ===================================
_________________________________ test_typical _________________________________

    def test_typical():
        # 1 normal (example): D=1200, S=50, H=2, expect about 244.949
>       assert calc_eoq(1200, 50, 2) == pytest.approx(244.949, rel=1e-3)

test_eoq.py:7: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

annual_demand = 1200, order_cost = 50, holding_cost = 2

    def calc_eoq(annual_demand, order_cost, holding_cost):
        """Economic order quantity: sqrt(2 * D * S / H).
    
        Raise ValueError if demand or order cost is negative,
        or if holding cost is zero or negative.
        """
>       raise NotImplementedError
E       NotImplementedError

inventory.py:7: NotImplementedError
__________________________________ test_large __________________________________

    def te

=========================== short test summary info ============================
FAILED test_eoq.py::test_typical - NotImplementedError
FAILED test_eoq.py::test_large - NotImplementedError
FAILED test_eoq.py::test_zero_demand - NotImplementedError
FAILED test_eoq.py::test_zero_holding_cost - NotImplementedError
FAILED test_eoq.py::test_negative_demand - NotImplementedError
5 failed in 0.14s


<div dir="rtl">

## שלב 4 - ירוק

עכשיו ממשו את `calc_eoq` בקובץ `inventory.py` (התא כותב את הקובץ מחדש): בדיקת הקלטים לפי המפרט, ואז הנוסחה. **אל תשנו את הבדיקות** (זו הרמאות הקלאסית: כשהבדיקה נכשלת, משנים אותה).

**כלים:** `math.sqrt(x)` מחשב שורש. תנאי: `if annual_demand < 0 or order_cost < 0:` ו-`if holding_cost <= 0:`. הנוסחה: `math.sqrt(2 * annual_demand * order_cost / holding_cost)`.

</div>

In [17]:
%%writefile inventory.py
import math


def calc_eoq(annual_demand, order_cost, holding_cost):
    """Economic order quantity: sqrt(2 * D * S / H).

    Raise ValueError if demand or order cost is negative,
    or if holding cost is zero or negative.
    """
    if annual_demand < 0 or order_cost < 0:
        raise ValueError("demand and order cost must not be negative")
    if holding_cost <= 0:
        raise ValueError("holding cost must be positive")
    return math.sqrt(2 * annual_demand * order_cost / holding_cost)

Overwriting inventory.py


In [18]:
!python -m pytest -q test_eoq.py

.....                                                                    [100%]


5 passed in 0.13s


In [19]:
# בדיקה - חלק ב', שלב 4: המימוש
def check_b2():
    mine = run_tests("test_eoq.py", "inventory.py")
    if "NotImplementedError" in mine["out"]:
        raise NotImplementedError
    assert mine["failed"] == 0 and mine["errors"] == 0, "הבדיקות שלכם עדיין נכשלות: " + ", ".join(failed_names(mine))
    oracle = run_tests("kit/oracle_eoq.py", "inventory.py")
    assert oracle["failed"] == 0 and oracle["errors"] == 0, \
        "בדיקות הייחוס של המורה נכשלו על המימוש שלכם: " + ", ".join(failed_names(oracle)) + " (רמז: חזרו למפרט - אילו קלטים אסורים?)"
    return True

check_b2() and print("✓ חלק ב' שלב 4: אדום ← ירוק, והמימוש עומד גם בבדיקות הייחוס")

✓ חלק ב' שלב 4: אדום ← ירוק, והמימוש עומד גם בבדיקות הייחוס


<div dir="rtl">

**(א)** למה חשוב לראות את הבדיקות נכשלות **לפני** שמממשים? מה היה קורה אילו הבדיקה הייתה עוברת כבר על הקוד הריק? (חשבו על "חיישן שמעולם לא ראיתם אותו מצפצף".)

**תשובה:** בלי כשל אין הוכחה שהבדיקה בודקת משהו: בדיקה ריקה (למשל assert True) או בדיקה שבודקת את הדבר הלא נכון עוברת תמיד. אדום ואז ירוק מוכיח שהמימוש הוא שגרם לשינוי. אילו הבדיקה עברה על קוד ריק, היה צריך לחשוד בה ולא לסמוך עליה.

</div>

<div dir="rtl">

---

# חלק ג' - מי בודק את הבדיקות? מוטנטים (8 דק')

## הרקע

כל הבדיקות שלכם ירוקות. האם זה אומר שהן **טובות**? בחלק א' ראינו בדיקה ירוקה וגם שגויה. איך מודדים איכות של בדיקות?

**משחק המוטציות:** שוברים את הקוד **בכוונה** (יוצרים "מוטנט": עותק של הפונקציה עם שינוי קטן), ומריצים עליו את הבדיקות שלכם.
- הבדיקות **נכשלו** ← תפסו את השבירה (`CAUGHT`). טוב.
- הבדיקות **עברו** על קוד שבור ← פספסו (`MISSED`). חסרה בדיקה.

זו בקרת איכות שבודקת את הבודק: מכניסים פריט פגום בכוונה ורואים אם הוא נתפס. **ציון ההרג** = כמה מתוך המוטנטים נתפסו.

בתיקייה `mutants/` יש **שישה** מוטנטים של `calc_eoq`:

| מוטנט | מה שונה |
|---|---|
| m1 | בלי הכפלה ב-2 בתוך השורש |
| m2 | בלי שורש |
| m3 | בלי בדיקת עלות החזקה |
| m4 | ביקוש 0 זורק שגיאה במקום להחזיר 0 |
| m5 | "הקשחה": מחזיר ערך מוכן רק לדוגמה אחת (1200, 50, 2), ובשאר המקרים מחשב לא נכון |
| m6 | בלי בדיקות קלט שליליות (ביקוש ועלות הזמנה) |

הנה אחד מהם, כדי לראות כמה קטן השינוי (השוו לפונקציה שכתבתם):

</div>

In [20]:
print(open("mutants/m1_no_factor_2.py").read())

import math


def calc_eoq(annual_demand, order_cost, holding_cost):
    if annual_demand < 0 or order_cost < 0:
        raise ValueError("demand and order cost must not be negative")
    if holding_cost <= 0:
        raise ValueError("holding cost must be positive")
    return math.sqrt(annual_demand * order_cost / holding_cost)



<div dir="rtl">

## שלב 1 - האם מוטנט נתפס?

ממשו `is_caught(result)`. הפונקציה מקבלת את תוצאת ההרצה של pytest (מילון עם `passed`, `failed`, `errors` - מספרי בדיקות) ומחזירה `True` אם המוטנט **נתפס**: כלומר **לפחות בדיקה אחת לא עברה** (`failed` גדול מ-0, או `errors` גדול מ-0).

שימו לב להיגיון ההפוך: כאן **כשל** של הבדיקות הוא הצלחה.

</div>

In [21]:
def is_caught(result):
    """result: {"passed": int, "failed": int, "errors": int}.
    Returns True if the mutant was caught: at least one test failed or crashed.
    e.g. is_caught({"passed": 3, "failed": 0, "errors": 0}) -> False   (the tests did not notice)
         is_caught({"passed": 1, "failed": 2, "errors": 0}) -> True
    """
    return result["failed"] > 0 or result["errors"] > 0

In [22]:
# בדיקה - חלק ג', שלב 1
def check_c1():
    assert is_caught({"passed": 3, "failed": 0, "errors": 0}) is False, "is_caught: כל הבדיקות עברו ← המוטנט לא נתפס (False)"
    assert is_caught({"passed": 1, "failed": 2, "errors": 0}) is True, "is_caught: יש כשלים ← המוטנט נתפס (True)"
    assert is_caught({"passed": 0, "failed": 0, "errors": 1}) is True, "is_caught: גם קריסה (errors) נחשבת כתפיסה"
    return True

check_c1() and print("✓ חלק ג' שלב 1: is_caught")

✓ חלק ג' שלב 1: is_caught


<div dir="rtl">

## שלב 2 - ציון הרג

ממשו `kill_score(statuses)`. היא מקבלת מילון `{שם מוטנט: True/False}` (`True` = נתפס) ומחזירה את **החלק שנתפס**: מספר בין 0 ל-1.

**כלים:** `sum(statuses.values())` סופר כמה ערכי `True` יש (True נחשב 1). `len(statuses)` - כמה מוטנטים.

</div>

In [23]:
def kill_score(statuses):
    """statuses: {mutant_name: True if caught else False}.
    Returns the fraction of mutants that were caught (0.0 - 1.0).
    e.g. kill_score({"a": True, "b": False, "c": True, "d": True}) -> 0.75
    """
    return sum(statuses.values()) / len(statuses)

In [24]:
# בדיקה - חלק ג', שלב 2
def check_c2():
    assert kill_score({"a": True, "b": False, "c": True, "d": True}) == 0.75, "kill_score: 3 מתוך 4 נתפסו ← 0.75"
    assert kill_score({"a": False, "b": False}) == 0, "kill_score: אף מוטנט לא נתפס ← 0"
    assert kill_score({"a": True}) == 1, "kill_score: כולם נתפסו ← 1"
    return True

check_c2() and print("✓ חלק ג' שלב 2: kill_score")

✓ חלק ג' שלב 2: kill_score


<div dir="rtl">

## שלב 3 - מריצים את הבדיקות שלכם נגד כל המוטנטים

הפונקציה `run_mutants` (נתונה, בדומה ל-`run_mutants.py` מההרצאה) עושה שלושה דברים: **(1)** מוודאת שהבדיקות עוברות על הקוד **הנכון** (בלי זה המדידה חסרת משמעות: בדיקה שנכשלת תמיד "תופסת" הכול), **(2)** מריצה את הבדיקות על כל קובץ ב-`mutants/`, **(3)** מחשבת ציון הרג בעזרת הפונקציות שכתבתם.

</div>

In [25]:
MUTANT_INFO = {
    "m1_no_factor_2": "בלי הכפלה ב-2 בתוך השורש",
    "m2_no_sqrt": "בלי שורש",
    "m3_no_holding_check": "בלי בדיקת עלות החזקה",
    "m4_zero_demand_error": "ביקוש 0 זורק שגיאה במקום להחזיר 0",
    "m5_hardcoded_example": "ערך מוכן לדוגמה אחת, ובשאר המקרים נוסחה שגויה",
    "m6_no_negative_check": "בלי בדיקות קלט שליליות (ביקוש ועלות הזמנה)",
}


def run_mutants(test_file, verbose=True):
    """Runs test_file on the correct code, and then on every mutant in mutants/.
    Returns {mutant_name: True (caught) / False (missed)}, or None if the tests do not even pass on the correct code."""
    base = run_tests(test_file, "kit/inventory_ref.py")
    if base["passed"] == 0 or base["failed"] or base["errors"]:
        print("הבדיקות לא עוברות על הקוד הנכון (או שאין בדיקות), ולכן אי אפשר למדוד אותן. תקנו קודם את הבדיקות:")
        print("   ", base["summary"], failed_names(base))
        return None
    statuses = {}
    for path in sorted(glob.glob("mutants/*.py")):
        name = os.path.basename(path)[:-3]
        statuses[name] = is_caught(run_tests(test_file, path))
        if verbose:
            if statuses[name]:
                print(f"{name:26s} CAUGHT")
            else:
                print(f"{name:26s} MISSED   <- {MUTANT_INFO.get(name, '')}")
    if verbose:
        caught = sum(statuses.values())
        print(f"\nציון הרג: {caught} מתוך {len(statuses)} ({kill_score(statuses):.0%})")
    return statuses

<div dir="rtl">

**לפני ההרצה:** לדעתכם, כמה מששת המוטנטים תפסו הבדיקות שכתבתם בשלב ב'? אילו לדעתכם יפספסו, ולמה? (רמז: שימו לב במה שונות הבדיקות שלכם מהטבלה במוטנטים m5 ו-m6.)

**התחזית שלי:** בדיקות כמו אלה תופסות את m1 ו-m2 (יש ערך מספרי מדויק), את m3 (בדיקה עם H=0), את m4 (בדיקת ביקוש 0) ואת m5 (בזכות test_large עם מספרים אחרים). הן עלולות לפספס את m6: בדיקה עם ביקוש שלילי בלבד עוברת גם בלי בדיקת הקלט, כי math.sqrt של מספר שלילי זורק ValueError בעצמו. התחזית הסבירה: 5 מתוך 6.

</div>

In [26]:
statuses = run_mutants("test_eoq.py")

m1_no_factor_2             CAUGHT
m2_no_sqrt                 CAUGHT


m3_no_holding_check        CAUGHT
m4_zero_demand_error       CAUGHT


m5_hardcoded_example       CAUGHT
m6_no_negative_check       MISSED   <- בלי בדיקות קלט שליליות (ביקוש ועלות הזמנה)

ציון הרג: 5 מתוך 6 (83%)


<div dir="rtl">

**(א)** אילו מוטנטים נתפסו ואילו פוספסו? בשביל כל אחד שפוספס: איזו בדיקה הייתה תופסת אותו? ומה זה אומר על בדיקות שמסתפקות ב"מקרה אחד מכל סוג"?

**תשובה:** אצלנו נתפסו m1-m5 ופוספס m6 (בלי בדיקות קלט שליליות). את m1, m2 ו-m5 תופסת בדיקה עם ערך מספרי מדויק על כמה קלטים (לא רק על דוגמה אחת), את m3 בדיקה עם H=0, ואת m4 בדיקת ביקוש 0. את m6 תופסת בדיקה שבה גם הביקוש וגם עלות ההזמנה שליליים, כי אז המכפלה חיובית ו-math.sqrt לא זורק שגיאה. בדיקות טובות נמדדות ביכולת לתפוס שינוי בקוד, לא בכך שהן עוברות.

</div>

<div dir="rtl">

**(ב)** בדיקת `test_negative_demand` שלכם (`calc_eoq(-5, 50, 2)`) עברה גם על m6, שהסיר את בדיקת הקלט. איך זה ייתכן? מה זה מלמד על הכלל "ירוק מהסיבות הנכונות"?

**תשובה:** עם ביקוש שלילי ועלות הזמנה חיובית, הביטוי שבתוך השורש שלילי, ו-math.sqrt זורק ValueError (math domain error) בעצמו. הבדיקה עברה, אבל לא בגלל בדיקת הקלט של הקוד אלא במקרה, מסיבה לא נכונה. ירוק לא מספיק: צריך לוודא שהוא ירוק מהסיבה שהתכוונו אליה, ומדידת מוטנטים חושפת את זה.

</div>

<div dir="rtl">

## שלב 4 - משפרים את הבדיקות עד שהכול נתפס

הוסיפו לקובץ `test_eoq.py` בדיקות שתופסות את המוטנטים שפוספסו. התא הבא **מוסיף** לסוף הקובץ (`%%writefile -a`), לא מוחק אותו. כתבו בדיקה אחת או יותר (שנו את השם והמחקו את `raise NotImplementedError`), הריצו, וחזרו על כך עד **6 מתוך 6**. אם כבר תפסתם הכול, הוסיפו בדיקה אחת של עלות הזמנה שלילית.

**חשוב:** כל בדיקה חדשה חייבת לעבור גם על הקוד הנכון (`run_mutants` יבדוק זאת תחילה).

</div>

In [27]:
%%writefile -a test_eoq.py


def test_both_negative():
    # both demand and order cost negative: the product is positive, so only an explicit check raises
    with pytest.raises(ValueError):
        calc_eoq(-5, -50, 2)

Appending to test_eoq.py


In [28]:
statuses = run_mutants("test_eoq.py")

m1_no_factor_2             CAUGHT
m2_no_sqrt                 CAUGHT


m3_no_holding_check        CAUGHT
m4_zero_demand_error       CAUGHT


m5_hardcoded_example       CAUGHT
m6_no_negative_check       CAUGHT

ציון הרג: 6 מתוך 6 (100%)


In [29]:
# בדיקה - חלק ג', שלב 4: ציון הרג 6/6
OFFICIAL = ["m1_no_factor_2", "m2_no_sqrt", "m3_no_holding_check",
            "m4_zero_demand_error", "m5_hardcoded_example", "m6_no_negative_check"]

def check_c3():
    base = run_tests("test_eoq.py", "kit/inventory_ref.py")
    if "NotImplementedError" in base["out"]:
        raise NotImplementedError
    result = run_mutants("test_eoq.py", verbose=False)
    assert result is not None, "הבדיקות ב-test_eoq.py נכשלות על הקוד הנכון (" + ", ".join(failed_names(base)) + ")"
    missed = [m for m in OFFICIAL if not result[m]]
    assert not missed, "עדיין לא נתפסו: " + ", ".join(f"{m} ({MUTANT_INFO[m]})" for m in missed)
    return True

check_c3() and print("✓ חלק ג' שלב 4: ציון הרג 6 מתוך 6")

✓ חלק ג' שלב 4: ציון הרג 6 מתוך 6


<div dir="rtl">

---

# חלק ד' - בדיקות שה-AI כתב: האם הן באמת בודקות? (6 דק')

## הרקע

בהרצאה למדנו שמרבית קוד הבדיקות ייכתב בעזרת AI, ושהסכנה היא **בדיקה מעגלית** או **חלשה**: הכול ירוק, אבל שום דבר לא נבדק. הקובץ `ai_tests.py` נכתב בכלי AI עבור `calc_eoq` (הוא נראה כמו מה שכלים כותבים הרבה פעמים). קראו אותו, והריצו אותו על הקוד הנכון:

</div>

In [30]:
print(open("ai_tests.py").read())

# Tests written by an AI assistant for calc_eoq (all of them pass on the correct code)
import pytest
from inventory import calc_eoq


def test_returns_number():
    result = calc_eoq(1200, 50, 2)
    assert isinstance(result, float)


def test_result_positive():
    assert calc_eoq(1200, 50, 2) > 0


def test_typical_value():
    result = calc_eoq(1200, 50, 2)
    assert result == pytest.approx(244.949, rel=0.5)


def test_zero_demand():
    result = calc_eoq(0, 50, 2)
    assert result is not None


def test_invalid_input():
    with pytest.raises(Exception):
        calc_eoq(1200, 50, 0)



In [31]:
print(run_tests("ai_tests.py", "kit/inventory_ref.py")["out"])

.....                                                                    [100%]
5 passed in 0.00s



<div dir="rtl">

כל חמש הבדיקות ירוקות על הקוד הנכון. **כמה מששת המוטנטים לדעתכם הן תופסות?** (קראו כל `assert`: מה בדיוק הוא בודק?)

**התחזית שלי:** מעט מאוד. test_returns_number ו-test_result_positive מקבלות כל מספר חיובי, test_typical_value עם rel=0.5 מקבלת כל ערך בין 122 ל-367 (כך גם m1, שנותן 173), test_zero_demand רק מוודא שחזר משהו, ו-test_invalid_input עם Exception הכללי מקבלת כל שגיאה. תפיסה צפויה: m2 ו-m4 בלבד, כלומר 2 מתוך 6.

</div>

In [32]:
ai_statuses = run_mutants("ai_tests.py")

m1_no_factor_2             MISSED   <- בלי הכפלה ב-2 בתוך השורש
m2_no_sqrt                 CAUGHT


m3_no_holding_check        MISSED   <- בלי בדיקת עלות החזקה
m4_zero_demand_error       CAUGHT


m5_hardcoded_example       MISSED   <- ערך מוכן לדוגמה אחת, ובשאר המקרים נוסחה שגויה
m6_no_negative_check       MISSED   <- בלי בדיקות קלט שליליות (ביקוש ועלות הזמנה)

ציון הרג: 2 מתוך 6 (33%)


<div dir="rtl">

**(א)** הבדיקות של ה-AI עברו על הקוד הנכון, ובכל זאת תפסו מעט מוטנטים. מה משותף ל-`assert`ים שלהן? למה ה-AI נוטה לכתוב כאלה, ואיך המפרט (כשנותנים אותו לפרומפט) משנה את זה?

**תשובה:** ה-assert חלש: הוא בודק סוג (float), סימן (> 0), שהוחזר משהו (is not None), טווח רחב מדי (rel=0.5) או שגיאה כללית (Exception), ולא את הערך המדויק שמגיע מהנוסחה. ה-AI כותב בדיקות שיעברו על כל מימוש סביר, ובלי ערכים צפויים ממפרט אין לו ציפייה מדויקת. כשנותנים לו טבלת מקרים עם תוצאות (ובקשה 'לפי המפרט, לא לפי הקוד'), הוא מתרגם אותן לבדיקות חזקות.

</div>

<div dir="rtl">

## שלב 1 - כלי קטן שמזהה assert חלש

לא צריך לחכות למדידה: אפשר לזהות **חשד** כבר מקריאת הקוד, כמו שכלי ניתוח (`ruff`) מוצא שאריות. ממשו `find_weak_asserts(source)`: היא מקבלת **טקסט** של קובץ בדיקות ומחזירה רשימה של זוגות `(מספר שורה, השורה)` לכל שורה שמכילה אחד מהדפוסים החלשים ב-`WEAK_PATTERNS`.

**כלים:** `for number, line in enumerate(source.splitlines(), 1):` נותן מספר שורה (מ-1) ואת השורה. `any(p in line for p in WEAK_PATTERNS)` בודק אם אחד הדפוסים מופיע בשורה. `line.strip()` מסיר רווחים בתחילת השורה.

**שימו לב:** זו **היוריסטיקה**, לא פסק דין. התבנית `> 0` (למשל) לגיטימית כשהדרישה היא בדיוק חיוביות, ולכן הפונקציה מסמנת שורות **לבדיקה אנושית** ולא פוסלת אותן. השאלה שתמיד שואלים: האם ה־`assert` הזה היה נכשל על קוד שגוי?

</div>

In [33]:
WEAK_PATTERNS = ["is not None", "isinstance(", "> 0", "raises(Exception)", "assert True", "rel=0.5"]


def find_weak_asserts(source):
    """source: the text of a test file.
    Returns a list of (line_number, stripped_line) for every line that contains one of WEAK_PATTERNS.
    Line numbers start at 1.
    e.g. find_weak_asserts("def test_a():\n    assert x is not None\n") -> [(2, "assert x is not None")]
    """
    found = []
    for number, line in enumerate(source.splitlines(), 1):
        if any(p in line for p in WEAK_PATTERNS):
            found.append((number, line.strip()))
    return found

In [34]:
# בדיקה - חלק ד', שלב 1
def check_d1():
    src = "def test_a():\n    assert x is not None\n\ndef test_b():\n    assert x == 5\n\ndef test_c():\n    assert r > 0\n"
    got = find_weak_asserts(src)
    assert got == [(2, "assert x is not None"), (8, "assert r > 0")], \
        "find_weak_asserts: בדוגמה צריכות להימצא שתי שורות: (2, ...) ו-(8, ...). מספור השורות מתחיל ב-1"
    assert len(find_weak_asserts(open("ai_tests.py", encoding="utf-8").read())) == 5, "find_weak_asserts: בקובץ של ה-AI כל חמש הבדיקות חלשות, צריכות להימצא 5 שורות"
    assert find_weak_asserts(open("kit/oracle_eoq.py", encoding="utf-8").read()) == [], "find_weak_asserts: בבדיקות הייחוס אין שורות חלשות"
    return True

check_d1() and print("✓ חלק ד' שלב 1: find_weak_asserts")

✓ חלק ד' שלב 1: find_weak_asserts


In [35]:
for number, line in find_weak_asserts(open("ai_tests.py", encoding="utf-8").read()):
    print(f"שורה {number}: {line}")
print()
print("הבדיקות שלי (test_eoq.py):", find_weak_asserts(open("test_eoq.py", encoding="utf-8").read()) or "לא נמצאו שורות חלשות")

שורה 8: assert isinstance(result, float)
שורה 12: assert calc_eoq(1200, 50, 2) > 0
שורה 17: assert result == pytest.approx(244.949, rel=0.5)
שורה 22: assert result is not None
שורה 26: with pytest.raises(Exception):

הבדיקות שלי (test_eoq.py): לא נמצאו שורות חלשות


<div dir="rtl">

## שלב 2 - צעד אמיתי: בקשו מעוזר AI בדיקות ל-`calc_eoq`

עד עכשיו עבדנו עם בדיקות מוכנות של AI. עכשיו נבדוק כלי אמיתי, **בלי לדעת מראש מה יצא**:

1. פתחו את **עוזר ה-AI של Colab** (כפתור ✨ **Gemini**, או "Generate" בתא קוד) - או כל כלי צ'אט שאתם משתמשים בו.
2. בקשו (העתיקו כמו שהוא; **שימו לב:** נתנו לו את ה-docstring כמפרט, בלי הקוד של המימוש):

```
Write pytest tests for the function below. Put them in a file that starts with
`from inventory import calc_eoq`. Follow the docstring, not the code.
Include normal, boundary and invalid cases.

def calc_eoq(annual_demand, order_cost, holding_cost):
    """Economic order quantity: sqrt(2 * D * S / H).

    Raise ValueError if demand or order cost is negative,
    or if holding cost is zero or negative.
    """
```

3. **הדביקו** את הבדיקות שהכלי כתב בתא הבא, **במקום** כל השורות שמתחת לשורה הראשונה (`%%writefile ...`). אם הכלי כתב גם קוד של הפונקציה עצמה - אל תדביקו אותו, רק את הבדיקות.
4. הריצו את התא שאחריו: הוא ימדוד את הבדיקות של הכלי (עוברות על הקוד הנכון? כמה מוטנטים תפסו? כמה שורות חלשות?).

> **אין לכם גישה ל-Gemini?** אפשר להשתמש בכל כלי צ'אט אחר (בדפדפן או בטלפון) ולהעתיק את התשובה, או לבקש מבן/בת זוג שיש לו/לה גישה. אם אין שום כלי: הדביקו את הבדיקות מהקובץ `ai_tests.py` ורשמו בתיעוד `"tool": "ללא כלי"`.

</div>

In [36]:
%%writefile ai_real_tests.py
# (example of what an AI tool may return)
import math
import pytest
from inventory import calc_eoq


def test_typical():
    assert calc_eoq(1200, 50, 2) == pytest.approx(math.sqrt(2 * 1200 * 50 / 2))


def test_known_value():
    assert calc_eoq(10000, 100, 5) == pytest.approx(632.456, rel=1e-3)


def test_zero_demand():
    assert calc_eoq(0, 50, 2) == 0


def test_negative_demand():
    with pytest.raises(ValueError):
        calc_eoq(-100, 50, 2)


def test_zero_holding_cost():
    with pytest.raises(ValueError):
        calc_eoq(1200, 50, 0)

Writing ai_real_tests.py


In [37]:
real_path = "ai_real_tests.py"
real_base = run_tests(real_path, "kit/inventory_ref.py")
real_n = real_base["passed"] + real_base["failed"] + real_base["errors"]
print(f"בדיקות שהכלי כתב: {real_n}, עברו על הקוד הנכון: {real_base['passed']}")
if real_base["failed"] or real_base["errors"]:
    print("ממצא: חלק מהבדיקות של הכלי נכשלות על הקוד הנכון (בדקו מול המפרט: הכלי טעה בציפייה?):", failed_names(real_base))
real_statuses = run_mutants(real_path)
real_weak = find_weak_asserts(open(real_path, encoding="utf-8").read())
print("שורות חשודות כחלשות:", len(real_weak))
for number, line in real_weak:
    print(f"   שורה {number}: {line}")

בדיקות שהכלי כתב: 5, עברו על הקוד הנכון: 5


m1_no_factor_2             CAUGHT
m2_no_sqrt                 CAUGHT


m3_no_holding_check        CAUGHT
m4_zero_demand_error       CAUGHT


m5_hardcoded_example       CAUGHT
m6_no_negative_check       MISSED   <- בלי בדיקות קלט שליליות (ביקוש ועלות הזמנה)

ציון הרג: 5 מתוך 6 (83%)
שורות חשודות כחלשות: 0


<div dir="rtl">

מלאו את התיעוד של הניסוי (העתיקו את המספרים שהודפסו למעלה):

</div>

In [38]:
real_log = {
    "tool": "Colab AI assistant",
    "tests": real_n,
    "passed_on_correct": real_base["passed"],
    "mutants_caught": sum(real_statuses.values()) if real_statuses else 0,
    "weak_lines": len(real_weak),
}

In [39]:
# בדיקה - חלק ד', שלב 2: תיעוד הצעד האמיתי (מאמת שתיעדתם נכון, לא מה שהכלי כתב)
def check_d2():
    assert real_log["tool"] != "(שם הכלי)", "real_log: כתבו את שם הכלי שבו השתמשתם"
    src = open("ai_real_tests.py", encoding="utf-8").read()
    if "paste the AI tests here" in src:
        raise NotImplementedError
    base = run_tests("ai_real_tests.py", "kit/inventory_ref.py")
    n = base["passed"] + base["failed"] + base["errors"]
    assert n >= 1, "ai_real_tests.py: לא נמצאו בדיקות (פונקציות שמתחילות ב-test_)"
    stat = run_mutants("ai_real_tests.py", verbose=False)
    caught = sum(stat.values()) if stat else 0
    weak = len(find_weak_asserts(src))
    for key in ("tests", "passed_on_correct", "mutants_caught", "weak_lines"):
        assert isinstance(real_log[key], int), f"real_log: הערך של {key} צריך להיות מספר שלם"
    assert real_log["tests"] == n, f"real_log: tests - הכלי כתב {n} בדיקות"
    assert real_log["passed_on_correct"] == base["passed"], f"real_log: passed_on_correct - על הקוד הנכון עברו {base['passed']}"
    assert real_log["mutants_caught"] == caught, f"real_log: mutants_caught - נתפסו {caught} מוטנטים (0 אם הבדיקות לא עברו על הקוד הנכון)"
    assert real_log["weak_lines"] == weak, f"real_log: weak_lines - נמצאו {weak} שורות חלשות"
    return True

check_d2() and print("✓ חלק ד' שלב 2: הניסוי מתועד")

✓ חלק ד' שלב 2: הניסוי מתועד


<div dir="rtl">

**(ב)** מה הכלי קיבל מכם ומה לא (הקוד? ה-docstring? הטבלה עם הערכים הצפויים?)? האם הבדיקות שלו עברו על הקוד הנכון, ומה ציון ההרג שלהן? אם היה לכם מפרט עם תוצאות מדויקות - מה הייתם מוסיפים לפרומפט כדי לשפר את התוצאה?

**תשובה:** הכלי קיבל רק את ה-docstring, בלי ערכים צפויים, ולכן נטה לחשב את הציפייה בעצמו (למשל math.sqrt(...) באותה נוסחה: בדיקה כמעט מעגלית) ולהשתמש בקלטים פשוטים. אם הבדיקות עברו על הקוד הנכון, זה עוד לא מוכיח שהן טובות: ציון ההרג הוא הראיה. הייתי מוסיף לפרומפט טבלת מקרים עם תוצאות מחושבות ביד ('1200, 50, 2 ← 244.95'), ומבקש כמה קלטים שונים ובדיקת קלט שלילי כפול.

</div>

<div dir="rtl">

---

# חלק ה' - סוקרים שינוי של סוכן (6 דק')

## הרקע

עד כה בדקנו אם הקוד **נכון**. כשסוכן משנה קוד, בודקים גם **מה הוא שינה**. לפני מיזוג שואלים חמש שאלות על ה-diff: **קבצים שלא התבקשו? התנהגות ששונתה בשקט? בדיקות שנמחקו או נחלשו? ערכים קבועים וסודות? מקרי קצה שחסרים?** (ובנוסף: **שאריות** וכל מה שכלי ניתוח כמו `ruff` מוצא.)

הסוכן קיבל משימה אחת: *"הוסף ל-`report.py` פונקציה שמחזירה את המוצרים שהמלאי שלהם נמוך מנקודת ההזמנה."* בתיקייה `lab_c/` יש את הקוד **לפני** ו**אחרי** השינוי, ואת ה-diff המלא. קראו אותו (שורות שמתחילות ב-`+` נוספו, ב-`-` נמחקו):

</div>

In [40]:
diff = open("lab_c/agent_change.diff", encoding="utf-8").read()
print(diff)

diff -ru before/inventory.py after/inventory.py
--- before/inventory.py	2026-10-16 20:28:07
+++ after/inventory.py	2026-10-16 20:28:07
@@ -14,4 +14,4 @@
         raise ValueError("demand and order cost must not be negative")
     if holding_cost <= 0:
         raise ValueError("holding cost must be positive")
-    return math.sqrt(2 * annual_demand * order_cost / holding_cost)
+    return round(math.sqrt(2 * annual_demand * order_cost / holding_cost))
diff -ru before/report.py after/report.py
--- before/report.py	2026-10-16 20:28:07
+++ after/report.py	2026-10-16 20:28:07
@@ -1,10 +1,22 @@
+import os
 import pandas as pd
 
+DEFAULT_PATH = "C:/Users/ran/orders.csv"
 
-def load_orders(path):
+
+def load_orders(path=DEFAULT_PATH):
     return pd.read_csv(path)
 
 
 def top_products(df, n=5):
     df = df.assign(revenue=df.quantity * df.unit_price)
     return df.groupby("product_name").revenue.sum().nlargest(n)
+
+
+def low_stock(stock, rop):
+    result = []
+    for i in range(len(stock

<div dir="rtl">

## שלב 1 - שני חיפושים אוטומטיים ב-diff

חמש השאלות מתחילות מדברים שאפשר לספור. ממשו שתי פונקציות שקוראות **טקסט של diff**:

- `changed_files(diff)` - רשימת **שמות הקבצים** ששונו, בסדר הופעתם. שורה שמתחילה ב-`"diff "` נראית כך: `diff -ru before/inventory.py after/inventory.py`. את הקובץ אפשר לקחת מהמילה האחרונה (`line.split()[-1]` נותן `after/inventory.py`), ואת השם בלי התיקייה עם `.split("/")[-1]`.
- `removed_tests(diff)` - רשימת **שמות הבדיקות שנמחקו**: שורות שמתחילות ב-`"-def test_"`. אם השורה היא `-def test_eoq_typical():` השם הוא `test_eoq_typical`. **רמז:** `line[len("-def "):]` חותך את ההתחלה, ו-`.split("(")[0]` לוקח עד הסוגר.

</div>

In [41]:
def changed_files(diff):
    """Returns the names (without the folder) of the files in a diff, in order.
    e.g. a line "diff -ru before/a.py after/a.py" gives "a.py"
    """
    files = []
    for line in diff.splitlines():
        if line.startswith("diff "):
            files.append(line.split()[-1].split("/")[-1])
    return files


def removed_tests(diff):
    """Returns the names of tests that the diff deleted: lines like "-def test_x():" give "test_x".
    """
    names = []
    for line in diff.splitlines():
        if line.startswith("-def test_"):
            names.append(line[len("-def "):].split("(")[0])
    return names

In [42]:
# בדיקה - חלק ה', שלב 1
def check_e1():
    assert changed_files("diff -ru before/a.py after/a.py\n+x\ndiff -ru before/t.py after/t.py\n") == ["a.py", "t.py"], "changed_files: שם הקובץ בלי התיקייה, בסדר ההופעה"
    assert removed_tests("-def test_a():\n+def test_b():\n-    x = 1\n--- before/f.py\n") == ["test_a"], "removed_tests: רק שורות שמתחילות ב--def test_ (לא שורות כמו ---)"
    assert changed_files(diff) == ["inventory.py", "report.py", "test_inventory.py"], "changed_files: על ה-diff של הסוכן צריכים לצאת שלושה קבצים"
    assert removed_tests(diff) == ["test_eoq_typical"], "removed_tests: בדיקה אחת נמחקה בקובץ של הסוכן"
    return True

check_e1() and print("✓ חלק ה' שלב 1:", changed_files(diff), "| בדיקות שנמחקו:", removed_tests(diff))

✓ חלק ה' שלב 1: ['inventory.py', 'report.py', 'test_inventory.py'] | בדיקות שנמחקו: ['test_eoq_typical']


<div dir="rtl">

**השוו למה שהתבקש:** הסוכן התבקש לשנות רק את `report.py`. ומה עם שאר הקבצים?

## שלב 2 - מריצים בדיקות על "לפני" ו"אחרי"

</div>

<div dir="rtl">

**לפני ההרצה:** כמה בדיקות יעברו בתיקייה `before` וכמה ב-`after`? והאם "הכול ירוק" אחרי השינוי מספיק כדי למזג? (חשבו על מה ש-`removed_tests` מצאה.)

**התחזית שלי:** ב-before: 4 passed. ב-after: 3 passed. שניהם ירוקים, אבל אחרי השינוי יש בדיקה אחת פחות: test_eoq_typical נמחקה, והיא הייתה נכשלת בגלל round() שהסוכן הוסיף ל-calc_eoq (245 במקום 244.949). ירוק אחרי מחיקת בדיקה הוא לא ראיה.

</div>

In [43]:
for folder in ("lab_c/before", "lab_c/after"):
    print(folder, "->", run_dir(folder)["summary"])

lab_c/before -> 4 passed in 0.00s


lab_c/after -> 3 passed in 0.00s


<div dir="rtl">

**(א)** מה אומר ההבדל בין `4 passed` ל-`3 passed`? ומה היה קורה אילו הבדיקה שנמחקה הייתה נשארת? (רמז: מה `round` עושה ל-244.949?)

**תשובה:** הסוכן 'תיקן' כשל על ידי מחיקתו: הבדיקה test_eoq_typical הייתה נכשלת כי round(244.949) נותן 245 והבדיקה דורשת קרוב ל-244.949. כשהיא נמחקה, pytest נשאר ירוק עם פחות בדיקות. מספר הבדיקות הוא רמז שקל לבדוק, ושינוי בקובץ בדיקות הוא הסימן החמור ביותר ב-diff.

</div>

<div dir="rtl">

## שלב 3 - רשימת ממצאים והחלטה

קראו שוב את ה-diff, ואת הקוד המלא של `lab_c/after/report.py` (הדפיסו אותו), וחפשו לפי רשימת התיוג **לפחות חמישה ממצאים** מסוגים שונים. כל ממצא הוא מילון עם שלושה שדות: `type`, `where` (קובץ ושורה/פונקציה), `problem` (משפט).

| `type` | משמעות | דוגמה |
|---|---|---|
| `"tests"` | בדיקה שנמחקה או נחלשה | _(הדוגמה למטה)_ |
| `"unrequested"` | קובץ שלא התבקש / התנהגות ששונתה בשקט | |
| `"hardcoded"` | ערך קבוע או סוד בקוד | |
| `"magic"` | מספר קסם בלי שם והסבר | |
| `"residue"` | ייבוא או משתנה שלא בשימוש | |
| `"edge"` | מקרה קצה חסר | |

</div>

In [44]:
print(open("lab_c/after/report.py").read())

import os
import pandas as pd

DEFAULT_PATH = "C:/Users/ran/orders.csv"


def load_orders(path=DEFAULT_PATH):
    return pd.read_csv(path)


def top_products(df, n=5):
    df = df.assign(revenue=df.quantity * df.unit_price)
    return df.groupby("product_name").revenue.sum().nlargest(n)


def low_stock(stock, rop):
    result = []
    for i in range(len(stock)):
        if stock.iloc[i]["on_hand"] < rop[stock.iloc[i]["product_id"]] * 1.1:
            result.append(stock.iloc[i]["product_id"])
    unused = len(result)
    return result



In [45]:
findings = [
    {"type": "tests", "where": "test_inventory.py", "problem": "בדיקה נמחקה (test_eoq_typical), ולכן pytest ירוק עם פחות בדיקות"},
    {"type": "unrequested", "where": "inventory.py", "problem": "קובץ שלא התבקש שונה: round() בתוך calc_eoq שינה את התוצאה ל-245"},
    {"type": "hardcoded", "where": "report.py:4", "problem": "נתיב קשיח של מחשב אישי: C:/Users/ran/orders.csv"},
    {"type": "magic", "where": "report.py, low_stock", "problem": "המספר 1.1 בלי שם והסבר"},
    {"type": "residue", "where": "report.py", "problem": "import os ומשתנה unused שלא בשימוש (ruff: F401, F841)"},
    {"type": "edge", "where": "report.py, low_stock", "problem": "אין טיפול ברשימה ריקה או במוצר שחסר ב-rop (KeyError), ואין בדיקה לפונקציה החדשה"},
]

decision = "request_changes"

In [46]:
# בדיקה - חלק ה', שלב 3
def check_e2():
    allowed = {"tests", "unrequested", "hardcoded", "magic", "residue", "edge"}
    assert len(findings) >= 5, "findings: רשמו לפחות חמישה ממצאים"
    for f in findings:
        assert set(f) == {"type", "where", "problem"}, "findings: בכל ממצא שלושה שדות בדיוק: type, where, problem"
        assert f["type"] in allowed, f"findings: type צריך להיות אחד מ-{sorted(allowed)}, ולא {f['type']!r}"
        assert len(f["where"].strip()) >= 3 and len(f["problem"].strip()) >= 8, "findings: מלאו where ו-problem (משפט קצר)"
    types = {f["type"] for f in findings}
    assert len(types) >= 5, f"findings: צריכים להיות ממצאים מחמישה סוגים שונים לפחות, יש {len(types)}"
    assert "tests" in types and "unrequested" in types, "findings: שני הממצאים החמורים ביותר הם בדיקה שנמחקה (tests) וקובץ שלא התבקש (unrequested)"
    assert decision in ("merge", "request_changes"), 'decision: כתבו "merge" או "request_changes"'
    assert decision == "request_changes", "decision: יש שינוי התנהגות שקט ובדיקה שנמחקה - ממזגים כך? חשבו שוב"
    return True

check_e2() and print("✓ חלק ה' שלב 3:", len(findings), "ממצאים, החלטה:", decision)

✓ חלק ה' שלב 3: 6 ממצאים, החלטה: request_changes


<div dir="rtl">

**(ב)** סוכן מדווח "כל הבדיקות ירוקות". מה שלוש הבדיקות הראשונות שתעשו לפני שתמזגו? ומה ההבדל בין מה ש-`ruff` (כלי ניתוח) תופס לבין מה ש-pytest תופס?

**תשובה:** בודקים (1) כמה בדיקות היו וכמה יש עכשיו, ומה השתנה בקבצי test_ ב-diff, (2) אילו קבצים שונו ואם ביקשנו אותם, (3) ערכים קבועים, מספרי קסם ומקרי קצה. ruff קורא קוד בלי להריץ אותו ותופס שאריות (ייבוא או משתנה שלא בשימוש), אבל לא יודע אם הקוד נכון: נוסחה שגויה לא תעורר בו אזהרה. pytest תופס טעויות לוגיות, ולכן הכלים משלימים זה את זה.

</div>

<div dir="rtl">

---

# חלק ו' - מסקנה (2 דק')

חמישה משפטים, אחד לכל חלק. כל משפט בנוי כך: **מה מימשתם ומדדתם ← איזה עיקרון זה מראה ← מה עושים בעקבות זה בעבודה עם AI.**

1. **בדיקה ראשונה:** כתבתי חמש בדיקות ממפרט ומצאתי ש-4 מהן נכשלות על קוד שנראה תקין, ולכן "רץ בלי שגיאה" לא מספיק, ובעבודה עם AI מבקשים בדיקות לפי המפרט ולא לפי הקוד.
2. **TDD:** ראיתי את כל הבדיקות אדומות על קוד ריק ואז ירוקות אחרי המימוש, וזה מוכיח שהן בודקות משהו, ולכן מבקשים מה-AI בדיקות קודם ומימוש אחר כך (בלי לשנות את הבדיקות).
3. **מוטנטים:** מדדתי ציון הרג ומצאתי שבדיקה עם ביקוש שלילי עברה גם בלי בדיקת קלט (m6), ולכן "ירוק" צריך להיות מהסיבה הנכונה, ומודדים בדיקות לפי כמה שבירות הן תופסות.
4. **בדיקות של AI:** בדיקות AI עם assert חלש עברו על הקוד הנכון ותפסו 2 מתוך 6 מוטנטים, ולכן לא מקבלים בדיקות של AI בלי למדוד אותן ולתת לו טבלת מקרים עם תוצאות ממפרט.
5. **סקירת שינוי של סוכן:** ב-diff של הסוכן מצאתי בדיקה שנמחקה, round() שהשתנה בשקט, נתיב קשיח, מספר קסם, שאריות ומקרה קצה, ולכן לפני מיזוג משווים מספר בדיקות, קבצים ששונו ותוכן ה-diff.

</div>

<div dir="rtl">

---

# בונוס

**1. מוטנט משלכם.** צרו קובץ `mutants/m7_....py` עם שינוי קטן משלכם ב-`calc_eoq` (למשל `holding_cost < 1` במקום `<= 0`, או החלפת `*` ב-`+`), והריצו שוב `run_mutants("test_eoq.py")`. האם הבדיקות שלכם תפסו אותו? אם לא - איזו בדיקה הייתה תופסת?

**2. טבלה במקום עותקים.** `@pytest.mark.parametrize("d,s,h,expected", [(1200, 50, 2, 244.949), (10000, 100, 5, 632.456)])` מאפשר לכתוב בדיקה אחת שרצה על כל שורה בטבלה. נסו להפוך את שתי בדיקות הערך שלכם לבדיקה אחת, והריצו את המוטנטים (האם הציון נשאר 6/6?).

**3. הבדיקה הראשונה לפרויקט הקבוצתי.** (זו גם משימת ההכנה למפגש 9.) בחרו את פונקציית הליבה בפרויקט שלכם, וכתבו שני מקרים לפי חמש הקטגוריות: אחד רגיל ואחד לא חוקי. מה הציפיות? מאיפה אתם יודעים שהן נכונות?

</div>

In [47]:
# תא לבונוס 1

In [48]:
# תא לבונוס 2

In [49]:
# תא לבונוס 3

<div dir="rtl">

---

# בדיקה מסכמת - הריצו לפני ההגשה

התא הבא מריץ שוב את כל תאי הבדיקה. לפני ההגשה: **Runtime ← Restart session and run all**, וודאו שכל השורות כאן מסומנות `✓` (חלקים ד' ו-ה' אפשר להשלים בבית).

</div>

In [50]:
for name, check in [("א' - חמש בדיקות ל-reorder", check_a1),
                    ("א' - תיקון calc_reorder_point", check_a2),
                    ("ב' - בדיקות EOQ כתובות (אדום)", check_b1),
                    ("ב' - מימוש calc_eoq (ירוק)", check_b2),
                    ("ג' - is_caught", check_c1),
                    ("ג' - kill_score", check_c2),
                    ("ג' - ציון הרג 6/6", check_c3),
                    ("ד' - find_weak_asserts", check_d1),
                    ("ד' - צעד אמיתי עם AI", check_d2),
                    ("ה' - changed_files, removed_tests", check_e1),
                    ("ה' - ממצאים והחלטה", check_e2)]:
    try:
        check()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"✗ {name}  ← עדיין לא מומש")
    except AssertionError as e:
        print(f"✗ {name}  ← {e}")
    except Exception as e:
        print(f"✗ {name}  ← {type(e).__name__}: {e}  (האם הרצתם את התא שכותב את הקובץ?)")

✓ א' - חמש בדיקות ל-reorder


✓ א' - תיקון calc_reorder_point


✓ ב' - בדיקות EOQ כתובות (אדום)


✓ ב' - מימוש calc_eoq (ירוק)
✓ ג' - is_caught
✓ ג' - kill_score


✓ ג' - ציון הרג 6/6
✓ ד' - find_weak_asserts


✓ ד' - צעד אמיתי עם AI
✓ ה' - changed_files, removed_tests
✓ ה' - ממצאים והחלטה
